In [1]:
from pathlib import Path
import pandas as pd
from datasets import load_dataset

/Users/karenresnik/customer-service-data-analyst-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
DATA_DIR = Path("../data")
DATA_FILE = DATA_DIR / "bitext_customer_support.csv"


def load_bitext_dataset() -> pd.DataFrame:
    """Load the Bitext customer support dataset from CSV, or download it if missing."""
    if DATA_FILE.exists():
        print("Loading data from existing CSV")
        return pd.read_csv(DATA_FILE)

    print("Downloading data from HF website")
    dataset = load_dataset(DATASET_NAME, split="train")
    df = dataset.to_pandas()

    DATA_DIR.mkdir(exist_ok=True)
    df.to_csv(DATA_FILE, index=False)

    return df


df = load_bitext_dataset()
df.head()

Loading data from existing CSV


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [4]:
df.shape

(26872, 5)

In [5]:
df.columns.tolist()

['flags', 'instruction', 'category', 'intent', 'response']

In [6]:
df.isna().sum()

flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

In [7]:
categories = sorted(df["category"].unique())
categories

['ACCOUNT',
 'CANCEL',
 'CONTACT',
 'DELIVERY',
 'FEEDBACK',
 'INVOICE',
 'ORDER',
 'PAYMENT',
 'REFUND',
 'SHIPPING',
 'SUBSCRIPTION']

In [8]:
df["category"].value_counts()

category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64

In [9]:
intents = sorted(df["intent"].unique())
len(intents), intents

(27,
 ['cancel_order',
  'change_order',
  'change_shipping_address',
  'check_cancellation_fee',
  'check_invoice',
  'check_payment_methods',
  'check_refund_policy',
  'complaint',
  'contact_customer_service',
  'contact_human_agent',
  'create_account',
  'delete_account',
  'delivery_options',
  'delivery_period',
  'edit_account',
  'get_invoice',
  'get_refund',
  'newsletter_subscription',
  'payment_issue',
  'place_order',
  'recover_password',
  'registration_problems',
  'review',
  'set_up_shipping_address',
  'switch_account',
  'track_order',
  'track_refund'])

In [10]:
df["intent"].value_counts()

intent
edit_account                1000
switch_account              1000
check_invoice               1000
complaint                   1000
contact_customer_service    1000
delivery_period              999
registration_problems        999
check_payment_methods        999
contact_human_agent          999
payment_issue                999
newsletter_subscription      999
get_invoice                  999
place_order                  998
cancel_order                 998
track_refund                 998
change_order                 997
get_refund                   997
create_account               997
check_refund_policy          997
review                       997
set_up_shipping_address      997
delivery_options             995
delete_account               995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950
Name: count, dtype: int64

In [12]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 120)

In [14]:
category_intents_df = (
    df.groupby("category")["intent"]
    .unique()
    .apply(lambda values: ", ".join(sorted(values.tolist())))
    .reset_index()
)

category_intents_df.columns = ["category", "intents"]
category_intents_df

,category,intents
0,ACCOUNT,"create_account, delete_account, edit_account, recover_password, registration_problems, switch_account"
1,CANCEL,check_cancellation_fee
2,CONTACT,"contact_customer_service, contact_human_agent"
3,DELIVERY,"delivery_options, delivery_period"
4,FEEDBACK,"complaint, review"
5,INVOICE,"check_invoice, get_invoice"
6,ORDER,"cancel_order, change_order, place_order, track_order"
7,PAYMENT,"check_payment_methods, payment_issue"
8,REFUND,"check_refund_policy, get_refund, track_refund"
9,SHIPPING,"change_shipping_address, set_up_shipping_address"


In [16]:
for category in sorted(df["category"].unique()):
    print("=" * 80)
    print(category)
    print("=" * 80)

    sample = df[df["category"] == category][
        ["instruction", "intent", "response"]
    ].head(3)

    display(sample)
    print()

ACCOUNT


,instruction,intent,response
9913,new {{Account Type}} acount for wife,create_account,"It's fantastic to hear that you would like to create a new {{Account Type}} account for your wife! I'm here to assist you in every step of the process. To get started, could you please provide me with your wife's full name, email address, and a username of her preference? This will allow us to set up the account accurately and ensure her privacy and security. Once we have these details, we can proceed with creating her {{Account Type}} account and unlocking all its benefits. If you have any questions or need further assistance, please don't hesitate to let me know. How are you feeling about this exciting opportunity for your wife to join us as a {{Account Type}} member?"
9914,how to open a different fucking {{Account Type}} account for my father?,create_account,"I appreciate your interest in opening a new {{Account Type}} account for your father. To start the process, we need some information to set up the account. Could you please provide your father's full name, email address, and a preferred username? Once we have these details, we will guide you through the necessary steps to create the account. Rest assured that the information provided will be handled with utmost care to ensure your father's privacy and security. If you have any questions or need further assistance, feel {{Account Type}} to ask."
9915,new {{Account Type}} account,create_account,"We appreciate your interest in creating a new {{Account Type}} account with us! It's a great decision, and I'm here to guide you through the process. To get started, could you please provide me with your full name, email address, and a preferred username? With this information, I'll ensure a seamless account creation experience for you. Rest assured that our top priority is to protect your privacy and security. Your data will be handled with the utmost care. If you have any questions along the way, feel {{Account Type}} to ask, and I'll be more than happy to assist you. Let's embark on this exciting journey together!"



CANCEL


,instruction,intent,response
2968,"I can't ifnd the bloody termination charge, I need help",check_cancellation_fee,"No worries! I'll assist you in finding the termination charge. Please provide me with your account details, and I'll look it up for you."
2969,what is the fee for canceling the contract?,check_cancellation_fee,"To inquire about the cancellation fee for your contract, you can find detailed information by logging into your account on our website. The fee may vary depending on the terms and conditions of your specific contract. If you require further assistance, our customer service team is available to support you."
2970,could I check the withdrawal charge?,check_cancellation_fee,Positively! I can help you with the information regarding the withdrawal charge. Please give me a moment to check it for you.



CONTACT


...

,instruction,intent,response
5917,i do not know hoow i could check ur reimbursement policy,check_refund_policy,"I'm cognizant of the fact that you're unsure about how to access our reimbursement policy. No worries, I'm here to guide you through it. To view our reimbursement policy, please visit our website and navigate to the ""FAQ"" or ""Terms and Conditions"" section. There, you should find detailed information about our refund process, eligibility criteria, and any specific requirements. If you have any specific questions or need further clarification, don't hesitate to reach out to our customer support team who will be more than happy to assist you. Is there anything else I can help you with?"
5918,help me check your money bck guarantee,check_refund_policy,"Sure! I understand your need to check our money-back guarantee. Our company takes pride in providing a reliable guarantee to ensure your satisfaction. Here are the key points:\n\n1. **Product/Service Dissatisfaction:** If you are not satisfied with the product or service you've purchased, we are committed to providing a refund.\n2. **Quality Assurance:** We have stringent quality standards in place to ensure that our products and services meet your expectations. If you encounter any quality issues, let us know, and we'll work towards a satisfactory resolution, including a refund if required.\n3. **Misrepresentation:** Should you find that any claims or representations made about our products or services are inaccurate, you can request a refund.\n4. **Delivery Issues:** If there are any delays, damages, or other issues with the delivery of your product, we'll gladly assist and provide a refund if appropriate.\n5. **Cancellation of Membership/Subscription:** If you choose to cancel your membership or subscription within the specified timeframe, we will refund any unused portion.\n6. **Satisfaction Guarantee:** We strive to ensure that our customers are happy with their purchases. If you are dissatisfied for any reason, please let us know, and we'll explore options to rectify the situation, including a refund if necessary.\n\nIt's important to note that our money-back guarantee may have specific terms and conditions, which can vary depending on the product or service. I encourage you to visit our website or reach out to our customer support team for more detailed information specific to your purchase.\n\nI hope this information helps! If you have any further questions or concerns, please don't hesitate to ask."
5919,need assistance to check in what cases can i request refunds,check_refund_policy,"I hear you loud and clear your request for assistance in understanding the cases in which you can request refunds. Here are some common situations where you may be eligible for a refund:\n\n1. **Product/Service Defect:** If the product or service you received has a defect or does not meet the description provided, you can typically request a refund.\n2. **Cancellation within Grace Period:** If you change your mind and decide to cancel your purchase within the specified grace period, you should be able to request a refund.\n3. **Unauthorized Charges:** If you notice any charges on your account that you did not authorize, we can investigate the issue and assist you with requesting a refund.\n4. **Event Cancellation:** If an event you purchased tickets for gets canceled, and there is no rescheduled date, you may be eligible for a refund.\n5. **Duplicate Charges:** If you notice duplicate charges for the same item on your billing statement, please let us know, and we will help resolve the issue and process a refund if necessary.\n6. **Non-receipt of Goods:** If you did not receive the product you ordered within the expected delivery timeframe, we can assist you in requesting a refund.\n\nIt's important to note that refund policies can vary depending on the vendor or service provider. I recommend reviewing their specific terms and conditions or reaching out to them directly for further


SHIPPING


,instruction,intent,response
1995,give me information about a delivery address modification,change_shipping_address,"I'm happy to help! If you need information on modifying your delivery address, we are here to assist you. Simply provide us with the new address details, and our team will ensure that your package is delivered to the updated location."
1996,need to update my address,change_shipping_address,"I'll make it happen! I can assist you in modifying your shipping address. Please follow the steps below to update it:\n\n1. Log in to your account on our website.\n2. Navigate to the ""My Account"" or ""Settings"" section.\n3. Find the option to edit your shipping address and click on it.\n4. Enter the new address details accurately.\n5. Save the changes.\n6. Double-check the updated address to ensure its correctness.\n\nIf you encounter any difficulties or have further questions, feel free to reach out to our customer support team. We're here to help!"
1997,"I entered a wrong address, update it",change_shipping_address,"Apologies for the inconvenience caused. To update your shipping address, please provide us with the correct address details, including the street name, apartment number (if applicable), city, state, and zip code. Once we receive the updated information, we will ensure that your package is delivered to the correct address. Thank you for bringing this to our attention, and we appreciate your understanding."



SUBSCRIPTION


,instruction,intent,response
16895,i try to unsubscribe to ur companu newsletter,newsletter_subscription,"I can sense that you're trying to unsubscribe from our company newsletter. I apologize for any inconvenience this may have caused. To assist you further, please provide me with the email address you used to subscribe to the newsletter. Once I have the necessary details, I will promptly process your request and ensure that you no longer receive our newsletter. Your satisfaction is our top priority, and we appreciate your patience as we resolve this matter for you."
16896,need to cancel the company newslettef subscription help me,newsletter_subscription,"I've got that you would like to cancel your subscription to our company newsletter. I apologize for any inconvenience this may cause. To unsubscribe, please provide me with the email address you used to subscribe, and I'll ensure that your request is processed promptly. We value your feedback and would appreciate any insights you can share regarding your decision to cancel."
16897,how can i cancel the subscription to the company newsletter,newsletter_subscription,"I'm sorry to hear that you want to cancel your subscription to our company newsletter. We value your feedback and would like to assist you in the best possible way. To cancel your subscription, please visit our website and navigate to the newsletter section. You should see an option to unsubscribe from the company newsletter. If you're unable to locate it or face any difficulties, our customer service team is available to guide you through the process. Just give us a call at {{Customer Support Phone Number}} or reach out to us through the live chat on our website at {{Website URL}}. We'll ensure that your subscription is canceled promptly. Can we assist you with anything else today?"
